# 01.4 — Temporal Structure: Periodicity, Dual FFT & MSTL Decomposition

## 1. Mục tiêu & Ranh giới Phương pháp luận
- **Phạm vi khảo sát (Zero Leakage)**: Tập Dev Train ($N = 6{,}329$, từ 2001-01-01 đến 2018-04-30). Toàn bộ Validation ($N=731$) và Test ($N=2{,}191$) được cách ly nghiêm ngặt.
- **Wavelet Removal**: Loại bỏ hoàn toàn Continuous Wavelet Transform (CWT) khỏi pipeline phân tích để triệt tiêu dead code và giảm tải bảo trì.
- **Bộ đôi Phổ tần số Dual FFT**:
  1. *Raw FFT*: Mean-centered trên toàn bộ chuỗi $N=6{,}329$, dải chu kỳ $[7, 730]$ ngày. Đẳng thức Parseval bảo toàn năng lượng ($\sum P \approx \mathrm{Var}(x)$) được đánh giá trên toàn dải với sai số số học.
  2. *Short-Term FFT*: Khử xu thế bằng trung bình trượt 30 ngày tâm đối xứng (loại 29 NaNs biên, $N_{\mathrm{valid}}=6{,}300$), đánh giá năng lượng trung bình bình phương thời gian, trích xuất các đỉnh dao động ngắn hạn trong dải $[7, 30]$ ngày.
- **Phân rã Đa mùa vụ MSTL**: Áp dụng trên thang đo $\log(1 + Y_t)$ với cấu hình cố định `periods=(7, 30, 365)`, `windows=(11, 15, 19)`, `iterate=2`. Tỷ số phương sai thành phần được định vị là chỉ số mô tả thuần túy (không trực giao, không bắt buộc tổng bằng 100%).
- **Quy ước Mùa khí hậu**: Cố định theo miền tri thức dự án: Mùa mưa (Tháng 5–11), Mùa khô (Tháng 12–4). Climatology theo cặp (Month, Day) với đường làm mượt tuần hoàn 15 ngày (circular wrap-around padding) triệt tiêu lệch pha năm nhuận.
- **Ranh giới Kiến trúc**: Chu kỳ thực nghiệm FFT $\\ne$ Chu kỳ ứng viên dự án $[7, 30, 122, 365] \\ne$ Siêu tham số mô hình hóa.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import scipy.signal as signal
from statsmodels.tsa.seasonal import MSTL

# Configure UTF-8 encoding
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

# Path resolution
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data.loader import DataLoader
from src.eda.TemporalAnalysis import TemporalStructureAnalyzer

# 1. Nạp dữ liệu Dev Train canonical (Zero Leakage)
dev_train, val_set, test_set = DataLoader().load_canonical_data(return_dev_train=True)
train_df = dev_train
print(f"📊 Phân tích Cấu trúc Thời gian (Dev Train: {len(train_df):,} dòng)")

# 2. Strict Assertion: Dev Train boundaries & continuity
assert len(train_df) == 6329, f"Expected 6,329 rows, got {len(train_df)}"
min_date = str(train_df['Ngày'].min().date())
max_date = str(train_df['Ngày'].max().date())
assert min_date == '2001-01-01', f"Expected min date 2001-01-01, got {min_date}"
assert max_date == '2018-04-30', f"Expected max date 2018-04-30, got {max_date}"

# Daily continuity check
date_diffs = train_df['Ngày'].diff().dropna()
assert (date_diffs.dt.days == 1).all(), "Daily continuity violated in Dev Train!"

# Leap-day verification: 4 leap years present in chronological sequence
leap_days = train_df[(train_df['Ngày'].dt.month == 2) & (train_df['Ngày'].dt.day == 29)]
print(f"📅 Leap-day count: {len(leap_days)} days ({[str(d.date()) for d in leap_days['Ngày']]})")
assert len(leap_days) == 4, f"Expected 4 leap days, got {len(leap_days)}"

# 3. Khởi tạo TemporalStructureAnalyzer
temp_analyzer = TemporalStructureAnalyzer(train_df, target_col='Lượng mưa', date_col='Ngày')
print("✅ Khởi tạo TemporalStructureAnalyzer thành công với chuẩn Zero-Leakage.")


📊 Phân tích Cấu trúc Thời gian (Dev Train: 6,329 dòng)
📅 Leap-day count: 4 days (['2004-02-29', '2008-02-29', '2012-02-29', '2016-02-29'])
✅ Khởi tạo TemporalStructureAnalyzer thành công với chuẩn Zero-Leakage.


In [2]:
# 1. Thực thi phân tích mùa vụ & lịch
seasonal_res = temp_analyzer.analyze_seasonal_patterns()
monthly_df = seasonal_res['monthly']
wet_months = seasonal_res['wet_months']
dry_months = seasonal_res['dry_months']

print("📅 QUY ƯỚC MÙA KHÍ HẬU DỰ ÁN (PROJECT DOMAIN CONVENTION):")
print(f"   • Các tháng mùa mưa (Wet Season):   Tháng {wet_months} (Tháng 5 đến Tháng 11)")
print(f"   • Các tháng mùa khô (Dry Season):   Tháng {dry_months} (Tháng 12 đến Tháng 4)")
print("\n📊 BẢNG THỐNG KÊ CHI TIẾT 12 THÁNG (DEV TRAIN N=6,329):")
display_cols = ['Month', 'count', 'mean', 'median', 'std', 'wet_day_pct_tau_1mm', 'positive_precip_pct']
display_df = monthly_df[display_cols].copy()
display_df.columns = ['Tháng', 'Số ngày', 'TB (mm)', 'Trung vị (mm)', 'Độ lệch chuẩn', 'Tỷ lệ mưa >= 1mm (%)', 'Tỷ lệ mưa > 0mm (%)']
print(display_df.to_string(index=False))


📅 QUY ƯỚC MÙA KHÍ HẬU DỰ ÁN (PROJECT DOMAIN CONVENTION):
   • Các tháng mùa mưa (Wet Season):   Tháng [5, 6, 7, 8, 9, 10, 11] (Tháng 5 đến Tháng 11)
   • Các tháng mùa khô (Dry Season):   Tháng [12, 1, 2, 3, 4] (Tháng 12 đến Tháng 4)

📊 BẢNG THỐNG KÊ CHI TIẾT 12 THÁNG (DEV TRAIN N=6,329):
 Tháng  Số ngày  TB (mm)  Trung vị (mm)  Độ lệch chuẩn  Tỷ lệ mưa >= 1mm (%)  Tỷ lệ mưa > 0mm (%)
     1      558   0.3415          0.000         1.8061                  6.45                37.46
     2      508   0.2529          0.000         2.7431                  3.35                22.05
     3      558   0.4253          0.000         1.6892                  7.71                35.30
     4      540   1.4368          0.020         5.5661                 19.81                60.00
     5      527   5.1761          1.340         9.1759                 55.60                93.55
     6      510   5.9974          2.825         8.2108                 70.20                96.27
     7      527   6.4796

In [3]:
# Trực quan hóa Seasonal Patterns & Climatology (4 panels)
fig_seasonal = temp_analyzer.plot_seasonal_patterns(seasonal_res, return_fig=True)
plt.show()


In [4]:
# Thực thi Dual FFT: Raw [7, 730] vs Short-Term [7, 30]
fft_res = temp_analyzer.analyze_fft(top_n=10, raw_period_range=(7, 730), short_term_period_range=(7, 30))
fft_raw = fft_res['fft_raw']
fft_short = fft_res['fft_short_term']

print("🔊 KẾT QUẢ DUAL FFT (SCIPY SIGNAL PERIODOGRAM):")
print("\n1. FFT RAW (Mean-Centered Only, Khảo sát dải [7, 730] ngày):")
diag_raw = fft_raw['energy_diagnostic']
print(f"   • Parseval Identity Check (Numerical Tolerance rtol=1e-5, atol=1e-8):")
print(f"     - Tổng công suất toàn phổ: {diag_raw['total_power_full_spectrum']:.6f}")
print(f"     - Phương sai mẫu thời gian: {diag_raw['sample_variance']:.6f}")
print(f"     - Sai số tuyệt đối:         {diag_raw['absolute_error']:.10f}")
print(f"     - Trạng thái kiểm tra:      {'PASSED ✅' if diag_raw['passed'] else 'FAILED ❌'}")
print(f"   • Độ rộng bin tần số:         {fft_raw['frequency_bin_width_cycles_per_day']:.7f} cycles/day")
print(f"   • Đỉnh ứng viên gần 365d:     {fft_raw['nearest_annual_peak']}")
print("   • Top 5 Đỉnh phổ Raw (chu kỳ dài / mùa vụ vĩ mô):")
for p in fft_raw['dominant_peaks'][:5]:
    print(f"     Rank {p['rank']}: Chu kỳ {p['period_days']} ngày (Tần số {p['frequency']}, Power {p['power']})")

print("\n2. FFT SHORT-TERM (30-Day Centered Rolling Mean Subtraction, Khảo sát dải [7, 30] ngày):")
diag_short = fft_short['energy_diagnostic']
print(f"   • Boundary Handling: 29 NaNs biên loại bỏ (15 đầu, 14 cuối), N_valid = {fft_short['preprocessing']['effective_sample_size']}")
print(f"   • Effective Date Range: {fft_short['effective_date_range']['start']} -> {fft_short['effective_date_range']['end']}")
print(f"   • Mean-Square Energy Check (Numerical Tolerance rtol=1e-5, atol=1e-8):")
print(f"     - Tổng công suất toàn phổ: {diag_short['total_power_full_spectrum']:.6f}")
print(f"     - Time Mean-Square Energy:  {diag_short['time_domain_mean_square']:.6f}")
print(f"     - Phương sai mẫu:           {diag_short['sample_variance']:.6f}")
print(f"     - Trung bình mẫu:           {diag_short['sample_mean']:.6f}")
print(f"     - Sai số tuyệt đối:         {diag_short['absolute_error']:.10f}")
print(f"     - Trạng thái kiểm tra:      {'PASSED ✅' if diag_short['passed'] else 'FAILED ❌'}")
print("   • Top 5 Đỉnh phổ Short-Term trong dải [7, 30] ngày (dao động ngắn hạn trong tháng):")
for p in fft_short['dominant_peaks'][:5]:
    print(f"     Rank {p['rank']}: Chu kỳ {p['period_days']} ngày (Tần số {p['frequency']}, Power {p['power']})")


🔊 KẾT QUẢ DUAL FFT (SCIPY SIGNAL PERIODOGRAM):

1. FFT RAW (Mean-Centered Only, Khảo sát dải [7, 730] ngày):
   • Parseval Identity Check (Numerical Tolerance rtol=1e-5, atol=1e-8):
     - Tổng công suất toàn phổ: 50.904990
     - Phương sai mẫu thời gian: 50.904990
     - Sai số tuyệt đối:         0.0000000000
     - Trạng thái kiểm tra:      PASSED ✅
   • Độ rộng bin tần số:         0.0001580 cycles/day
   • Đỉnh ứng viên gần 365d:     {'rank': 1, 'period_days': 372.29, 'frequency': 0.002686, 'power': 5.080089}
   • Top 5 Đỉnh phổ Raw (chu kỳ dài / mùa vụ vĩ mô):
     Rank 1: Chu kỳ 372.29 ngày (Tần số 0.002686, Power 5.080089)
     Rank 2: Chu kỳ 121.71 ngày (Tần số 0.008216, Power 0.498461)
     Rank 3: Chu kỳ 421.93 ngày (Tần số 0.00237, Power 0.178219)
     Rank 4: Chu kỳ 575.36 ngày (Tần số 0.001738, Power 0.159516)
     Rank 5: Chu kỳ 20.29 ngày (Tần số 0.049297, Power 0.145027)

2. FFT SHORT-TERM (30-Day Centered Rolling Mean Subtraction, Khảo sát dải [7, 30] ngày):
   • Bound

In [5]:
# Trực quan hóa đối chiếu phổ tần số Dual FFT (2 panels)
fig_fft = temp_analyzer.plot_spectrum(fft_res, return_fig=True)
plt.show()


In [6]:
# 1. Phân rã MSTL Đa tầng với Explicit Windows (11, 15, 19) trên log1p
mstl_periods = (7, 30, 365)
mstl_windows = (11, 15, 19)
mstl_res = temp_analyzer.decompose_mstl(periods=mstl_periods, windows=mstl_windows, iterate=2)

print("📊 MSTL MULTI-SEASONAL DECOMPOSITION (STATSMODELS 0.15.0):")
print(f"   • Biến đổi target:              {mstl_res['transform_method']} (z = log(1 + y))")
print(f"   • Chu kỳ phân rã:               {mstl_res['periods']} (Tuần: 7d, Tháng: 30d, Năm: 365d)")
print(f"   • Requested Windows:            {mstl_res['requested_windows']}")
print(f"   • Resolved Windows:             {mstl_res['resolved_windows']}")
print(f"   • Iterations:                   {mstl_res['iterate']}")
print(f"   • Ghi chú chu kỳ s=30:          30-day sub-seasonal / intra-month recurring component (not MJO)")

print("\n📈 TỶ LỆ PHƯƠNG SAI MÔ TẢ (DESCRIPTIVE COMPONENT VARIANCE RATIOS):")
print("   (Ghi chú: Thành phần MSTL không trực giao, tổng các tỷ lệ không bắt buộc bằng 100%)")
var_ratios = mstl_res['variance_ratios']
for comp, ratio in var_ratios.items():
    print(f"   • {comp:15s}: {ratio:.4f} ({ratio*100:.2f}%)")
print(f"   • Tổng tỷ lệ:      {sum(var_ratios.values()):.4f} ({sum(var_ratios.values())*100:.2f}%)")

# 2. Handoff Contract verification cho 01.5
resid = mstl_res['resid']
assert isinstance(resid, pd.Series), "resid must be pd.Series"
assert isinstance(resid.index, pd.DatetimeIndex), "resid.index must be pd.DatetimeIndex"
assert resid.index.equals(train_df.set_index('Ngày').index), "resid index must strictly equal train_df index"
assert len(resid) == 6329, f"resid length must be 6,329, got {len(resid)}"
assert resid.dtype == np.float64, "resid dtype must be float64"
assert np.isfinite(resid).all(), "resid must have all finite values"
print("\n✅ MSTL Residual Handoff Contract cho 01.5: VERIFIED (Exact Index Equality & Finite Float64)")

# 3. Trực quan hóa phân rã MSTL (6 panels)
fig_mstl = temp_analyzer.plot_decomposition(mstl_res, return_fig=True)
plt.show()


📊 MSTL MULTI-SEASONAL DECOMPOSITION (STATSMODELS 0.15.0):
   • Biến đổi target:              log1p (z = log(1 + y))
   • Chu kỳ phân rã:               [7, 30, 365] (Tuần: 7d, Tháng: 30d, Năm: 365d)
   • Requested Windows:            [11, 15, 19]
   • Resolved Windows:             [11, 15, 19]
   • Iterations:                   2
   • Ghi chú chu kỳ s=30:          30-day sub-seasonal / intra-month recurring component (not MJO)

📈 TỶ LỆ PHƯƠNG SAI MÔ TẢ (DESCRIPTIVE COMPONENT VARIANCE RATIOS):
   (Ghi chú: Thành phần MSTL không trực giao, tổng các tỷ lệ không bắt buộc bằng 100%)
   • trend          : 0.0113 (1.13%)
   • residual       : 0.3981 (39.81%)
   • seasonal_7     : 0.0530 (5.30%)
   • seasonal_30    : 0.0474 (4.74%)
   • seasonal_365   : 0.4254 (42.54%)
   • Tổng tỷ lệ:      0.9352 (93.52%)

✅ MSTL Residual Handoff Contract cho 01.5: VERIFIED (Exact Index Equality & Finite Float64)


In [7]:
# 1. Tổng hợp bằng chứng cấu trúc thời gian & ranh giới phương pháp luận
print("=" * 80)
print("📌 TỔNG HỢP KẾT LUẬN & RANH GIỚI PHƯƠNG PHÁP LUẬN (01.4 TEMPORAL STRUCTURE):")
print("   1. Quần thể nghiên cứu: Dev Train N=6,329 (2001-01-01 -> 2018-04-30), Zero-Leakage.")
print("   2. Quy ước mùa khí hậu: Wet Season = Tháng 5-11, Dry Season = Tháng 12-4.")
print("   3. Đẳng thức bảo toàn năng lượng Parseval trên FFT: Verified qua Numerical Tolerance.")
print("   4. Phân rã MSTL đa tầng: periods=(7, 30, 365), windows=(11, 15, 19) explicit, scale log1p.")
print("   5. Ranh giới kiến trúc: Chu kỳ thực nghiệm FFT != Chu kỳ ứng viên dự án [7, 30, 122, 365] != Model Parameters.")
print("=" * 80)

# 2. Chuẩn bị Canonical Artifact Schema
OUTPUTS_DIR = ROOT_DIR / "reports" / "eda"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
canonical_report_path = OUTPUTS_DIR / "01.4_temporal_structure_report.json"

BRIDGE_DIR = ROOT_DIR / "notebooks" / "eda" / "outputs"
BRIDGE_DIR.mkdir(parents=True, exist_ok=True)
bridge_report_path = BRIDGE_DIR / "temporal_structure_report.json"

DATA_OUTPUT_DIR = ROOT_DIR / "data" / "processed" / "eda_outputs"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Monthly climatology data dict
monthly_climatology_data = {}
for _, row in monthly_df.iterrows():
    m_str = str(int(row['Month']))
    monthly_climatology_data[m_str] = {
        'mean': round(float(row['mean']), 4),
        'median': round(float(row['median']), 4),
        'std': round(float(row['std']), 4),
        'wet_day_pct_tau_1mm': round(float(row['wet_day_pct_tau_1mm']), 2),
        'positive_precip_pct': round(float(row['positive_precip_pct']), 2)
    }

canonical_report = {
    "contract_metadata": {
        "module": "01.4_temporal_structure",
        "analysis_population": "dev_train",
        "sample_size": 6329,
        "date_range": {
            "start": "2001-01-01",
            "end": "2018-04-30"
        },
        "zero_leakage_guarantee": "Validation (2018-05-01 to 2020-04-30, N=731) and Test (2020-05-01 to 2026-04-30, N=2191) strictly excluded.",
        "leap_day_policy": {
            "canonical_series_handling": "All 4 leap days (2004-02-29, 2008-02-29, 2012-02-29, 2016-02-29) retained in chronological sequence without modification.",
            "mstl_representation": "Fitted on continuous native daily grid (N=6,329) with integer period s=365. Represents a fixed 365-observation periodic cycle; calendar phase drift (+4 days over 17.3 years, 0.063%) is acknowledged as a known structural limitation of fixed-period discrete MSTL diagnostic. No claim of automatic Gregorian phase correction is made.",
            "annual_climatology_representation": "Aggregated by (Month, Day) calendar tuples to prevent post-February phase drift; Feb 29 computed across its 4 leap occurrences; 15-day moving average applies circular wrap-around boundary handling."
        },
        "environment_versions": {
            "pandas": str(pd.__version__),
            "numpy": str(np.__version__),
            "scipy": str(signal.__version__ if hasattr(signal, '__version__') else 'scipy'),
            "statsmodels": '0.15.0'
        }
    },
    "climate_seasons": {
        "definition_type": "domain_fixed_project_convention",
        "convention_basis": "project_domain_convention",
        "wet_season_months": [5, 6, 7, 8, 9, 10, 11],
        "dry_season_months": [12, 1, 2, 3, 4],
        "usage_boundary": "Descriptive seasonal regime grouping only; not a mandatory modeling feature."
    },
    "monthly_climatology": {
        "schema_description": "Dict mapping month string ('1'..'12') to statistical summaries",
        "month_entry_schema": {
            "mean": "float",
            "median": "float",
            "std": "float",
            "wet_day_pct_tau_1mm": "float",
            "positive_precip_pct": "float"
        },
        "data": monthly_climatology_data
    },
    "fft_raw_spectrum": {
        "preprocessing": fft_raw['preprocessing'],
        "energy_diagnostic": fft_raw['energy_diagnostic'],
        "period_range_evaluated": [7, 730],
        "period_range_semantic": "inclusive_range [7, 730]",
        "frequency_bin_width_cycles_per_day": fft_raw['frequency_bin_width_cycles_per_day'],
        "peak_selection_semantics": "top-N local spectral bins ranked by power; neighboring bins may reflect spectral leakage/side lobes and are not assumed to be mutually independent periodicities",
        "dominant_peaks": fft_raw['dominant_peaks'],
        "annual_signal_diagnostic": {
            "candidate_target_period_days": 365,
            "nearest_detected_bin_period_days": fft_raw['nearest_annual_peak']['period_days'] if fft_raw['nearest_annual_peak'] else None,
            "interpretation": "Empirical diagnostic evidence of low-frequency concentration in the annual-scale band; represents exploratory observation, not a deterministic modeling parameter."
        }
    },
    "fft_short_term_diagnostic": {
        "preprocessing": fft_short['preprocessing'],
        "energy_diagnostic": fft_short['energy_diagnostic'],
        "peak_selection_band": [7, 30],
        "peak_selection_band_semantic": "inclusive_range [7, 30]",
        "frequency_bin_width_cycles_per_day": fft_short['frequency_bin_width_cycles_per_day'],
        "peak_selection_semantics": "top-N local spectral bins ranked by power within configured [7, 30] band",
        "dominant_peaks": fft_short['dominant_peaks'],
        "technical_note": "Peaks reflect short-period intra-monthly variations restricted to [7, 30] days; longer periodic signals are attenuated by the 30-day centered moving average filter."
    },
    "mstl_decomposition": {
        "target_transformation": {
            "method": "log1p",
            "formula": "z = log(1 + y)",
            "rationale": "Preserves zero values naturally and compresses heavy right tail; used as a descriptive analysis scale in 01.4, does not enforce log transformation in downstream modeling."
        },
        "configuration": {
            "periods": list(mstl_res['periods']),
            "requested_windows": list(mstl_res['requested_windows']),
            "resolved_windows": list(mstl_res['resolved_windows']),
            "iterate": int(mstl_res['iterate']),
            "statsmodels_version": "0.15.0"
        },
        "period_semantic_labels": {
            "7": "weekly / calendar-scale seasonality",
            "30": "30-day sub-seasonal / intra-month recurring component (not MJO)",
            "365": "annual-scale macro seasonality"
        },
        "variance_diagnostics": {
            "diagnostic_type": "relative_component_variance_descriptive",
            "interpretation_note": "Ratios indicate marginal relative dispersion Var(C)/Var(z); components are non-orthogonal, so ratios do not represent orthogonal variance attribution and do not necessarily sum to 100% due to cross-component covariances.",
            "component_variance_ratios": {k: round(float(v), 6) for k, v in mstl_res['variance_ratios'].items()}
        }
    },
    "period_candidates_for_downstream_modeling": {
        "source_type": "project_domain_hypotheses",
        "candidate_periods_days": [7, 30, 122, 365],
        "candidate_descriptions": {
            "7": "Weekly calendar-scale recurrence hypothesis",
            "30": "Sub-seasonal / intra-month recurrence hypothesis",
            "122": "Quarterly / seasonal macro recurrence hypothesis (domain hypothesis; NOT fitted in MSTL)",
            "365": "Annual macro monsoon cycle hypothesis"
        },
        "conceptual_boundary": "Observed FFT Peaks != Project Candidate Periods != Final Model Seasonal Parameters. Final model parameters must be independently validated during modeling stages."
    },
    "mstl_residual_handoff_contract": {
        "consumer": "01.5_stationarity_autocorrelation",
        "accepted_object_type": "pd.Series",
        "accepted_index_type": "pd.DatetimeIndex",
        "index_equals_train_index": True,
        "expected_length": 6329,
        "finite_values_required": True,
        "semantic_nature": "EDA-derived diagnostic series only; NOT a modeling feature."
    },
    "transitional_bridge_artifact": {
        "status": "temporary_compatibility_infrastructure",
        "path": "notebooks/eda/outputs/temporal_structure_report.json",
        "purpose": "Maintains backwards compatibility for unrefactored 01.7 notebook.",
        "lifecycle_roadmap": {
            "phase_1_current": "Generated alongside canonical report to prevent breaking 01.7.",
            "phase_2_migration": "Update 01.7 to read directly from reports/eda/01.4_temporal_structure_report.json.",
            "phase_3_exit": "Permanently delete notebooks/eda/outputs/temporal_structure_report.json post-01.7 migration."
        }
    },
    "wavelet_analysis_status": {
        "status": "removed_from_module",
        "scope": "TemporalAnalysis.py",
        "rationale": "Continuous Wavelet Transform (~175-182d) has no operational downstream usage; code and pywt imports removed from module to eliminate dead code and maintenance burden."
    }
}

# Export canonical report
with open(canonical_report_path, "w", encoding="utf-8") as f:
    json.dump(canonical_report, f, indent=2, ensure_ascii=False)
print(f"\n💾 Canonical artifact exported to: {canonical_report_path}")

# Export transitional bridge report (with compatible top-level fields for 01.7)
bridge_report = {
    "temporal_methods": "Dual FFT + MSTL (Continuous Wavelet excluded from primary pipeline)",
    "wavelet_exclusion_rationale": "CWT ~175-182d periodicity has no downstream operational modeling usage; MSTL provides comprehensive multi-seasonal decomposition with lower complexity.",
    "candidate_domain_periods": [7, 30, 122, 365],
    "fft_detected_periods": [int(round(p['period_days'])) for p in fft_raw['dominant_peaks']],
    "mstl_periods": [7, 30, 365],
    "wet_season_months": [5, 6, 7, 8, 9, 10, 11],
    "dry_season_months": [12, 1, 2, 3, 4],
    "monthly_mean_rainfall_mm": {str(k): round(float(v['mean']), 4) for k, v in monthly_climatology_data.items()},
    "period_distinction_note": "Observed FFT Peaks != Project Candidate Periods != Final Model Seasonal Parameters.",
    "canonical_artifact_pointer": "reports/eda/01.4_temporal_structure_report.json"
}
with open(bridge_report_path, "w", encoding="utf-8") as f:
    json.dump(bridge_report, f, indent=2, ensure_ascii=False)
print(f"💾 Transitional bridge artifact exported to: {bridge_report_path}")

# Also save to data/processed/eda_outputs for legacy backup
with open(DATA_OUTPUT_DIR / "temporal_structure_report.json", "w", encoding="utf-8") as f:
    json.dump(bridge_report, f, indent=2, ensure_ascii=False)


📌 TỔNG HỢP KẾT LUẬN & RANH GIỚI PHƯƠNG PHÁP LUẬN (01.4 TEMPORAL STRUCTURE):
   1. Quần thể nghiên cứu: Dev Train N=6,329 (2001-01-01 -> 2018-04-30), Zero-Leakage.
   2. Quy ước mùa khí hậu: Wet Season = Tháng 5-11, Dry Season = Tháng 12-4.
   3. Đẳng thức bảo toàn năng lượng Parseval trên FFT: Verified qua Numerical Tolerance.
   4. Phân rã MSTL đa tầng: periods=(7, 30, 365), windows=(11, 15, 19) explicit, scale log1p.
   5. Ranh giới kiến trúc: Chu kỳ thực nghiệm FFT != Chu kỳ ứng viên dự án [7, 30, 122, 365] != Model Parameters.

💾 Canonical artifact exported to: D:\DS108_project\reports\eda\01.4_temporal_structure_report.json
💾 Transitional bridge artifact exported to: D:\DS108_project\notebooks\eda\outputs\temporal_structure_report.json
